# Q1D | Train a Linear Regression Model

# Imports

In [35]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

from Q1.consts import *

from Q1.utils import (
	read_csv_to_df,

	PREPROCESSING,
	PIPELINES,
	SCORING
)

from sklearn.linear_model import (
	LinearRegression,
	SGDRegressor,
	Ridge, RidgeCV,
	Lasso, LassoCV,
	ElasticNet,
)

from sklearn.pipeline import Pipeline


# Fetch Data

In [36]:
train_df = read_csv_to_df(path=TRAIN_PATH)

X = train_df.drop(columns=TARGET)
y = train_df[TARGET]

# CV Folds

- Every model in this notebook is scored on the same 20 folds: 4-fold CV repeated over the 5 seeds in `FOLD_SEEDS`.
- Validation folds hold complete rows only, so pipelines that drop rows and pipelines that impute are validated on the exact same rows. Incomplete rows only ever join the training side, for pipelines that keep them.
- Every pipeline is cloned and seeded with `SEED` (every `random_state`) by `PIPELINES.evaluate_pipeline`.

In [37]:
FOLDS = PIPELINES.make_folds(X, seeds=FOLD_SEEDS, n_splits=4)

print(f"Folds: {len(FOLDS)}")
print(f"Complete rows (validation): {X.notna().all(axis=1).sum():,}")
print(f"Incomplete rows (training only): {X.isna().any(axis=1).sum():,}")

Folds: 20
Complete rows (validation): 1,374
Incomplete rows (training only): 185


# Linear Regression

## Linear Regression | BASE

In [38]:
LR_BASE_results = {}

LR_DROPPEDROWS_Xclean, LR_DROPPEDROWS_yclean = PREPROCESSING.drop_rows_with_missing_data(
		X=X.copy(), y=y.copy(),
)

In [39]:
LR_BASE_Pipeline = Pipeline([
	('impute', 'passthrough'),
	('transform', 'passthrough'),
	('scale', 'passthrough'),
	('model', LinearRegression())
])

PIPELINES.display_pipeline_grid({
    ("Linear Regression BASE", "BASE: drop missing rows"): LR_BASE_Pipeline,
})

In [40]:
LR_BASE_results["BASE"] = PIPELINES.evaluate_pipeline(
	LR_BASE_Pipeline, X=LR_DROPPEDROWS_Xclean, y=LR_DROPPEDROWS_yclean, folds=FOLDS,
)

SCORING.results_df(LR_BASE_results, baseline="BASE")

,rmse mean,rmse sd,mae mean,mae sd,r2 mean,r2 sd,Δrmse vs BASE,Δrmse SE,Δ / SE,p,verdict,folds
intervention,,,,,,,,,,,,
BASE,0.546,0.024,0.422,0.018,0.765,0.02,0.0,NaN,NaN,NaN,baseline,20


## Stochastic Gradient Descent (SGD) | BASE

In [41]:
SGD_results = {}

SGD_DROPPEDROWS_Xclean, SGD_DROPPEDROWS_yclean = PREPROCESSING.drop_rows_with_missing_data(
		X=X.copy(), y=y.copy(),
)

In [42]:
SGD_BASE_Pipeline = Pipeline([
	('impute', 'passthrough'),
	('transform', 'passthrough'),
	('scale', 'passthrough'),
	('model', SGDRegressor(penalty=None))
])

PIPELINES.display_pipeline_grid({
    ("SGD_BASE", "BASE: drop missing rows"): SGD_BASE_Pipeline,
})

SGD_StandardScaler_Pipeline = Pipeline([
	('impute', 'passthrough'),
	('transform', 'passthrough'),
	('scale', StandardScaler()),
	('model', SGDRegressor(penalty=None))
])

In [43]:
SGD_results["BASE"] = PIPELINES.evaluate_pipeline(
	SGD_BASE_Pipeline, X=SGD_DROPPEDROWS_Xclean, y=SGD_DROPPEDROWS_yclean, folds=FOLDS,
)

SGD_results["StandardScaler"] = PIPELINES.evaluate_pipeline(
	SGD_StandardScaler_Pipeline, X=SGD_DROPPEDROWS_Xclean, y=SGD_DROPPEDROWS_yclean, folds=FOLDS,
)

SCORING.results_df(SGD_results, baseline="BASE")

,rmse mean,rmse sd,mae mean,mae sd,r2 mean,r2 sd,Δrmse vs BASE,Δrmse SE,Δ / SE,p,verdict,folds
intervention,,,,,,,,,,,,
StandardScaler,5.460000e-01,2.400000e-02,4.220000e-01,1.800000e-02,7.640000e-01,2.000000e-02,-3.679916e+11,3.138915e+11,-1.172,0.128,candidate,20
BASE,3.679916e+11,5.069803e+11,3.645744e+11,5.032148e+11,-2.991050e+23,5.166882e+23,0.000000e+00,NaN,NaN,NaN,baseline,20


## Stochastic Gradient Descent (SGD) | $penalty = l_2$

In [10]:
SGD_BASE_l2_results = {}

In [11]:
SGD_BASE_l2_Xclean, SGD_BASE_l2_yclean = PREPROCESSING.drop_rows_with_missing_data(
		X=X.copy(), y=y.copy(),
)

SGD_BASE_l2_Pipeline = Pipeline([
	('impute', 'passthrough'),
	('transform', 'passthrough'),
	('scale', 'passthrough'),
	('model', SGDRegressor(penalty='l2'))
])

PIPELINES.display_pipeline_grid({
    ("SGD_BASE_l2 BASE", "drop missing rows"): SGD_BASE_l2_Pipeline,
})

In [12]:
SGD_BASE_l2_results["BASE"] = PIPELINES.evaluate_pipeline(
	SGD_BASE_l2_Pipeline, X=SGD_BASE_l2_Xclean, y=SGD_BASE_l2_yclean, folds=FOLDS,
)

SCORING.results_df(SGD_BASE_l2_results, baseline="BASE")

,rmse mean,rmse sd,mae mean,mae sd,r2 mean,r2 sd,Δrmse vs BASE,Δrmse SE,Δ / SE,p,verdict,folds
intervention,,,,,,,,,,,,
BASE,2.394908e+11,3.015059e+11,2.373084e+11,2.993488e+11,-1.082505e+23,2.614079e+23,0.0,NaN,NaN,NaN,baseline,20


## Stochastic Gradient Descent (SGD) | $penalty = l_1$

In [13]:
SGD_BASE_l1_results = {}

In [14]:
SGD_BASE_l1_Xclean, SGD_BASE_l1_yclean = PREPROCESSING.drop_rows_with_missing_data(
		X=X.copy(), y=y.copy(),
)

SGD_BASE_l1_Pipeline = Pipeline([
	('impute', 'passthrough'),
	('transform', 'passthrough'),
	('scale', 'passthrough'),
	('model', SGDRegressor(penalty='l1'))
])

PIPELINES.display_pipeline_grid({
    ("SGD_BASE_l1 BASE", "drop missing rows"): SGD_BASE_l1_Pipeline,
})

In [15]:
SGD_BASE_l1_results["BASE"] = PIPELINES.evaluate_pipeline(
	SGD_BASE_l1_Pipeline, X=SGD_BASE_l1_Xclean, y=SGD_BASE_l1_yclean, folds=FOLDS,
)

SCORING.results_df(SGD_BASE_l1_results, baseline="BASE")

,rmse mean,rmse sd,mae mean,mae sd,r2 mean,r2 sd,Δrmse vs BASE,Δrmse SE,Δ / SE,p,verdict,folds
intervention,,,,,,,,,,,,
BASE,2.593630e+11,3.034540e+11,2.571915e+11,3.013920e+11,-1.213639e+23,2.520796e+23,0.0,NaN,NaN,NaN,baseline,20


## Stochastic Gradient Descent (SGD) | $penalty = elasticnet$

In [16]:
SGD_BASE_elasticnet_results = {}

In [17]:
SGD_BASE_elasticnet_Xclean, SGD_BASE_elasticnet_yclean = PREPROCESSING.drop_rows_with_missing_data(
		X=X.copy(), y=y.copy(),
)

SGD_BASE_elasticnet_Pipeline = Pipeline([
	('impute', 'passthrough'),
	('transform', 'passthrough'),
	('scale', 'passthrough'),
	('model', SGDRegressor(penalty='elasticnet'))
])

PIPELINES.display_pipeline_grid({
    ("SGD_BASE_elasticnet BASE", "drop missing rows"): SGD_BASE_elasticnet_Pipeline,
})

In [18]:
SGD_BASE_elasticnet_results["BASE"] = PIPELINES.evaluate_pipeline(
	SGD_BASE_elasticnet_Pipeline, X=SGD_BASE_elasticnet_Xclean, y=SGD_elasticnet_yclean, folds=FOLDS,
)

SCORING.results_df(SGD_BASE_elasticnet_results, baseline="BASE")

,rmse mean,rmse sd,mae mean,mae sd,r2 mean,r2 sd,Δrmse vs BASE,Δrmse SE,Δ / SE,p,verdict,folds
intervention,,,,,,,,,,,,
BASE,2.894886e+11,3.231209e+11,2.868949e+11,3.209795e+11,-1.462893e+23,2.632776e+23,0.0,NaN,NaN,NaN,baseline,20


## Ridge Regression | BASE

In [19]:
RR_BASE_results = {}

In [20]:
RR_BASE_Xclean, RR_BASE_yclean = PREPROCESSING.drop_rows_with_missing_data(
		X=X.copy(), y=y.copy(),
)

RR_BASE_Pipeline = Pipeline([
	('impute', 'passthrough'),
	('transform', 'passthrough'),
	('scale', 'passthrough'),
	('model', Ridge())
])

PIPELINES.display_pipeline_grid({
    ("Ridge Regression BASE", "drop missing rows"): RR_BASE_Pipeline,
})

In [21]:
RR_BASE_results["BASE"] = PIPELINES.evaluate_pipeline(
	RR_BASE_Pipeline, X=RR_BASE_Xclean, y=RR_BASE_yclean, folds=FOLDS,
)

SCORING.results_df(RR_BASE_results, baseline="BASE")

,rmse mean,rmse sd,mae mean,mae sd,r2 mean,r2 sd,Δrmse vs BASE,Δrmse SE,Δ / SE,p,verdict,folds
intervention,,,,,,,,,,,,
BASE,0.546,0.025,0.423,0.018,0.764,0.02,0.0,NaN,NaN,NaN,baseline,20


## Ridge Regression CV | Base

In [22]:
RRcv_BASE_results = {}

In [23]:
RRcv_BASE_Xclean, RRcv_BASE_yclean = PREPROCESSING.drop_rows_with_missing_data(
		X=X.copy(), y=y.copy(),
)

RRcv_BASE_Pipeline = Pipeline([
	('impute', 'passthrough'),
	('transform', 'passthrough'),
	('scale', 'passthrough'),
	('model', RidgeCV())
])

PIPELINES.display_pipeline_grid({
    ("Ridge Regression CV BASE", "drop missing rows"): RRcv_BASE_Pipeline,
})

In [24]:
RRcv_BASE_results["BASE"] = PIPELINES.evaluate_pipeline(
	RRcv_BASE_Pipeline, X=RRcv_BASE_Xclean, y=RRcv_BASE_yclean, folds=FOLDS,
)

SCORING.results_df(RRcv_BASE_results, baseline="BASE")

,rmse mean,rmse sd,mae mean,mae sd,r2 mean,r2 sd,Δrmse vs BASE,Δrmse SE,Δ / SE,p,verdict,folds
intervention,,,,,,,,,,,,
BASE,0.546,0.024,0.422,0.018,0.765,0.02,0.0,NaN,NaN,NaN,baseline,20


## Lasso Regression | BASE

In [25]:
L_BASE_results = {}

In [26]:
L_BASE_Xclean, L_BASE_yclean = PREPROCESSING.drop_rows_with_missing_data(
		X=X.copy(), y=y.copy(),
)

L_BASE_Pipeline = Pipeline([
	('impute', 'passthrough'),
	('transform', 'passthrough'),
	('scale', 'passthrough'),
	('model', Lasso())
])

PIPELINES.display_pipeline_grid({
    ("Lasso Regression BASE", "drop missing rows"): L_BASE_Pipeline,
})

In [27]:
L_BASE_results["BASE"] = PIPELINES.evaluate_pipeline(
	L_BASE_Pipeline, X=L_BASE_Xclean, y=L_BASE_yclean, folds=FOLDS,
)

SCORING.results_df(L_BASE_results, baseline="BASE")

,rmse mean,rmse sd,mae mean,mae sd,r2 mean,r2 sd,Δrmse vs BASE,Δrmse SE,Δ / SE,p,verdict,folds
intervention,,,,,,,,,,,,
BASE,0.746,0.025,0.61,0.022,0.561,0.022,0.0,NaN,NaN,NaN,baseline,20


## Lasso Regression CV | BASE

In [28]:
Lcv_BASE_results = {}

In [29]:
Lcv_BASE_Xclean, Lcv_BASE_yclean = PREPROCESSING.drop_rows_with_missing_data(
		X=X.copy(), y=y.copy(),
)

Lcv_BASE_Pipeline = Pipeline([
	('impute', 'passthrough'),
	('transform', 'passthrough'),
	('scale', 'passthrough'),
	('model', LassoCV())
])

PIPELINES.display_pipeline_grid({
    ("Lasso Regression CV BASE", "drop missing rows"): Lcv_BASE_Pipeline,
})

In [30]:
Lcv_BASE_results["BASE"] = PIPELINES.evaluate_pipeline(
	Lcv_BASE_Pipeline, X=Lcv_BASE_Xclean, y=Lcv_BASE_yclean, folds=FOLDS,
)

SCORING.results_df(Lcv_BASE_results, baseline="BASE")

,rmse mean,rmse sd,mae mean,mae sd,r2 mean,r2 sd,Δrmse vs BASE,Δrmse SE,Δ / SE,p,verdict,folds
intervention,,,,,,,,,,,,
BASE,0.553,0.025,0.434,0.019,0.759,0.021,0.0,NaN,NaN,NaN,baseline,20


## Elastic Net | BASE

In [31]:
EN_BASE_results = {}

In [32]:
EN_BASE_Xclean, EN_BASE_yclean = PREPROCESSING.drop_rows_with_missing_data(
		X=X.copy(), y=y.copy(),
)

EN_BASE_Pipeline_BASE = Pipeline([
	('impute', 'passthrough'),
	('transform', 'passthrough'),
	('scale', 'passthrough'),
	('model', ElasticNet())
])

PIPELINES.display_pipeline_grid({
    ("Elastic Net BASE", "drop missing rows"): EN_BASE_Pipeline_BASE,
})

In [33]:
EN_BASE_results["BASE"] = PIPELINES.evaluate_pipeline(
	EN_BASE_Pipeline_BASE, X=EN_BASE_Xclean, y=EN_BASE_yclean, folds=FOLDS,
)

SCORING.results_df(EN_BASE_results, baseline="BASE")

,rmse mean,rmse sd,mae mean,mae sd,r2 mean,r2 sd,Δrmse vs BASE,Δrmse SE,Δ / SE,p,verdict,folds
intervention,,,,,,,,,,,,
BASE,0.738,0.024,0.604,0.022,0.57,0.023,0.0,NaN,NaN,NaN,baseline,20


---

# Combined Results

In [34]:
combined_results = {
	f"{model} | {name}": scores
	for model, results in {
		"Linear Regression": LR_results,
		"SGD": SGD_results,
		"Ridge Regression": RR_results, "Ridge Regression CV": RRcv_results,
		"Lasso Regression": L_results, "Lasso Regression CV": Lcv_results,
		"Elastic Net": EN_results,
	}.items()
	for name, scores in results.items()
}

combined_df = SCORING.results_df(combined_results, baseline="Linear Regression | BASE", holm=True)
combined_df

,rmse mean,rmse sd,mae mean,mae sd,r2 mean,r2 sd,Δrmse vs Linear Regression | BASE,Δrmse SE,Δ / SE,p,p (Holm),verdict,folds
intervention,,,,,,,,,,,,,
Ridge Regression CV | BASE,5.460000e-01,2.400000e-02,4.220000e-01,1.800000e-02,7.650000e-01,2.000000e-02,-0.000000e+00,0.000000e+00,-0.301,0.383,1.0,within noise,20
Linear Regression | BASE,5.460000e-01,2.400000e-02,4.220000e-01,1.800000e-02,7.650000e-01,2.000000e-02,0.000000e+00,NaN,NaN,NaN,NaN,baseline,20
Ridge Regression | BASE,5.460000e-01,2.500000e-02,4.230000e-01,1.800000e-02,7.640000e-01,2.000000e-02,0.000000e+00,1.000000e-03,0.180,0.570,1.0,within noise,20
Lasso Regression CV | BASE,5.530000e-01,2.500000e-02,4.340000e-01,1.900000e-02,7.590000e-01,2.100000e-02,7.000000e-03,4.000000e-03,1.777,0.954,1.0,within noise,20
Elastic Net | BASE,7.380000e-01,2.400000e-02,6.040000e-01,2.200000e-02,5.700000e-01,2.300000e-02,1.930000e-01,1.200000e-02,15.840,1.000,1.0,within noise,20
Lasso Regression | BASE,7.460000e-01,2.500000e-02,6.100000e-01,2.200000e-02,5.610000e-01,2.200000e-02,2.010000e-01,1.300000e-02,15.752,1.000,1.0,within noise,20
SGD | BASE,3.679916e+11,5.069803e+11,3.645744e+11,5.032148e+11,-2.991050e+23,5.166882e+23,3.679916e+11,3.138915e+11,1.172,0.872,1.0,within noise,20
